In [ ]:
# Task 1
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = {
    'square_footage': [1200, 1500, 1800, 2000, 2200, 1400, np.nan, 1700, 2100, 1600, 1900, 2300, 2500, 1300, 1750],
    'bedrooms': [2, 3, 3, 4, 4, 3, 2, np.nan, 4, 3, 3, 5, 5, 2, 3],
    'bathrooms': [1, 2, 2, 3, 3, 2, 1, 2, np.nan, 2, 2, 4, 3, 1, 2],
    'age': [15, 10, 8, 5, 4, 12, 20, 9, 6, 11, 7, 3, 2, 18, np.nan],
    'neighborhood': ['A', 'B', 'B', 'C', 'C', 'A', 'A', 'B', 'C', np.nan, 'B', 'C', 'C', 'A', 'B'],
    'price': [180000, 240000, 280000, 330000, 360000, 220000, 160000, 265000, 345000, 250000, 295000, 410000, 450000, 175000, 275000]
}

df = pd.DataFrame(data)

df['square_footage'] = df['square_footage'].fillna(df['square_footage'].median())
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())
df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median())
df['age'] = df['age'].fillna(df['age'].median())
df['neighborhood'] = df['neighborhood'].fillna(df['neighborhood'].mode()[0])

X = df.drop('price', axis=1)
y = df['price']

X_encoded = pd.get_dummies(X, columns=['neighborhood'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

importance_df = pd.DataFrame({'feature': X_encoded.columns, 'importance': model.feature_importances_}).sort_values(by='importance', ascending=False)

print('Cleaned Dataset:')
print(df)
print('')
print('Feature Importance:')
print(importance_df)
print('')
print('MAE:', mae)
print('RMSE:', rmse)
print('R2 Score:', r2)

new_house = pd.DataFrame([{
    'square_footage': 1950,
    'bedrooms': 4,
    'bathrooms': 3,
    'age': 6,
    'neighborhood': 'B'
}])

new_house_encoded = pd.get_dummies(new_house, columns=['neighborhood'], drop_first=True)
new_house_encoded = new_house_encoded.reindex(columns=X_encoded.columns, fill_value=0)

predicted_price = model.predict(new_house_encoded)
print('')
print('Predicted Price for New House:', predicted_price[0])

In [ ]:
# Task 2
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

emails = pd.DataFrame({
    'email_text': [
        'Win a free iPhone now click this link',
        'Meeting tomorrow at 10 AM in office',
        'Limited offer buy now and get 50 percent off',
        'Please find the attached project report',
        'Congratulations you have won cash prize claim now',
        'Can we reschedule our call to next week',
        'Get cheap meds with guaranteed delivery click here',
        'Lunch at 1 PM lets discuss the assignment',
        'Urgent your account is compromised verify immediately',
        'Thank you for your payment receipt attached',
        'Exclusive deal for you click and earn rewards',
        'Team update the sprint demo is on Friday',
        'Earn money fast from home no experience needed',
        'Final reminder submit tax details to avoid penalty',
        'You are selected for lottery claim your reward'
    ],
    'length': [42, 36, 52, 41, 58, 40, 55, 44, 57, 45, 50, 43, 54, 56, 53],
    'has_hyperlink': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    'sender': [
        'promo@dealz.com', 'manager@company.com', 'sale@offers.com', 'colleague@company.com',
        'lottery@winz.com', 'hr@company.com', 'meds@cheaprx.com', 'friend@university.edu',
        'security@alert-mail.com', 'billing@trustedshop.com', 'rewards@clicknow.com', 'lead@company.com',
        'income@quickcash.com', 'gov@taxoffice.org', 'prize@luckydraw.com'
    ],
    'spam': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
})

X = emails[['email_text', 'length', 'has_hyperlink', 'sender']]
y = emails['spam']

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(), 'email_text'),
        ('sender', OneHotEncoder(handle_unknown='ignore'), ['sender']),
        ('num', 'passthrough', ['length', 'has_hyperlink'])
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print('Evaluation Metrics:')
print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 Score:', f1)
print('Confusion Matrix:')
print(cm)

def classify_new_email(email_text, length, has_hyperlink, sender):
    new_email = pd.DataFrame([{
        'email_text': email_text,
        'length': length,
        'has_hyperlink': has_hyperlink,
        'sender': sender
    }])
    prediction = model.predict(new_email)[0]
    probability = model.predict_proba(new_email)[0][1]
    return prediction, probability

new_prediction, new_probability = classify_new_email(
    'Congratulations claim your free vacation now click the link',
    60,
    1,
    'promo@dealz.com'
 )

print('')
print('New Email Prediction (1=Spam, 0=Not Spam):', new_prediction)
print('Spam Probability:', new_probability)

In [ ]:
# Task 3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

customer_df = pd.DataFrame({
    'total_spending_6m': [1200, 4500, 300, 8000, 2200, 1500, 9800, 400, 6000, 3500, np.nan, 500, 7200, 1800, 2500, 9200, 1300, 4700, 11000, 900],
    'age': [22, 45, 28, 50, 35, 31, 48, 24, 42, 39, 29, np.nan, 46, 33, 37, 52, 27, 41, 55, 26],
    'number_of_visits': [4, 14, 2, 20, 8, 6, 21, 1, 16, 12, 5, 2, 18, 7, 9, 22, 4, 13, 25, 3],
    'purchase_frequency': [1.2, 3.8, 0.6, 4.5, 2.1, 1.8, 4.7, 0.5, 3.9, 3.0, 1.4, 0.7, 4.2, 1.9, 2.4, 4.8, 1.3, 3.4, 5.0, 0.9],
    'high_value': [0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0]
})

customer_df['total_spending_6m'] = customer_df['total_spending_6m'].fillna(customer_df['total_spending_6m'].median())
customer_df['age'] = customer_df['age'].fillna(customer_df['age'].median())

feature_cols = ['total_spending_6m', 'age', 'number_of_visits', 'purchase_frequency']
for col in feature_cols:
    q1 = customer_df[col].quantile(0.25)
    q3 = customer_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    customer_df[col] = customer_df[col].clip(lower, upper)

X = customer_df[feature_cols]
y = customer_df['high_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_precision = precision_score(y_test, svm_pred, zero_division=0)
svm_recall = recall_score(y_test, svm_pred, zero_division=0)
svm_f1 = f1_score(y_test, svm_pred, zero_division=0)
svm_cm = confusion_matrix(y_test, svm_pred)

coef = svm_model.coef_[0]
intercept = svm_model.intercept_[0]

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_pred)
tree_precision = precision_score(y_test, tree_pred, zero_division=0)
tree_recall = recall_score(y_test, tree_pred, zero_division=0)
tree_f1 = f1_score(y_test, tree_pred, zero_division=0)
tree_cm = confusion_matrix(y_test, tree_pred)

rules = export_text(tree_model, feature_names=feature_cols)

print('Cleaned Dataset:')
print(customer_df)
print('')
print('SVM Separating Hyperplane:')
print('w:', coef)
print('b:', intercept)
print('Equation: w1*x1 + w2*x2 + w3*x3 + w4*x4 + b = 0')
print('')
print('Decision Rules:')
print(rules)
print('')
print('SVM Metrics:')
print('Accuracy:', svm_accuracy)
print('Precision:', svm_precision)
print('Recall:', svm_recall)
print('F1 Score:', svm_f1)
print('Confusion Matrix:')
print(svm_cm)
print('')
print('Decision Tree Metrics:')
print('Accuracy:', tree_accuracy)
print('Precision:', tree_precision)
print('Recall:', tree_recall)
print('F1 Score:', tree_f1)
print('Confusion Matrix:')
print(tree_cm)